# Does removing the 9-frame clips matter?

`cutoff_9` drops every clip with 9 or fewer frames. Because of the off-by-one in the labels made
before 2026-10-05 (see [WLASL_info.md](../../info/WLASL_info.md), "Gotchas" and "Removed by
`cutoff_9`"), that was 3 clips in WLASL-2000 and 2 in WLASL-1000, and a 0-based `cutoff_9` would
drop only `15144`. All 5 are real clips of 9 or 10 frames. This notebook asks whether dropping
them changes any result, so the decision whether to regenerate `cutoff_9` rests on evidence.

Three things are checked:

1. **Where the clips sit:** which sets and classes they are in, and what share of each set they are.
2. **How trained models treat them:** the SATNAC 2026 and MViTv2_B_32x3 checkpoints, which were
   trained without them, are run on the clips, with both label conventions.
3. **What they would have done to the reported numbers:** the validation accuracy and loss of
   those runs with the clips added back. The test sets are unaffected, since no removed clip is
   in a test set.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.results import STASH_DIR_NAME, load_json
from src.run_types import LABEL_SUFFIX, LABELS_PATH, RAW_DIR
from src.utils import decoded_frame_count
from src.visualise2 import set_thesis_style

set_thesis_style()

clip_ids = ["18223", "59958", "15144"]  # the clips cutoff_9 removed, by video id
split_stems = ["asl1000", "asl2000"]
scores_stash_p = Path(STASH_DIR_NAME) / "short_clip_scores.json"
noise_stash_p = Path(STASH_DIR_NAME) / "short_clip_noise.json"


def load_labels(split: str) -> pd.DataFrame:
    """Every instance of a label directory, with a `set` column."""
    frames = []
    for set_name in ("train", "val", "test"):
        df = pd.read_json(LABELS_PATH / split / f"{set_name}_{LABEL_SUFFIX}", dtype={"video_id": str})
        frames.append(df.assign(set=set_name))
    return pd.concat(frames, ignore_index=True)

## The clips

`start`/`end` are the label's `frame_start`/`frame_end`. A label loads `frames[start:end]`, so
`length` is what the model gets. The 1-based labels skip each clip's first frame
(`*_1_indexed`, which the trained models saw); the 0-based labels keep it.

In [2]:
one_based, zero_based = load_labels("asl2000_1_indexed"), load_labels("asl2000")
rows = []
for video_id in clip_ids:
    old = one_based[one_based.video_id == video_id].iloc[0]
    new = zero_based[zero_based.video_id == video_id].iloc[0]
    same_class = zero_based[zero_based.label_num == new.label_num].set.value_counts()
    rows.append(
        {
            "video_id": video_id,
            "gloss": new.label_name,
            "set": new.set,
            "signer": new.signer_id,
            "clip_frames": decoded_frame_count(next(RAW_DIR.glob(f"{video_id}.*"))),
            "1-based length": old.frame_end - old.frame_start,
            "0-based length": new.frame_end - new.frame_start,
            "class train/val/test": "/".join(str(same_class.get(s, 0)) for s in ("train", "val", "test")),
        }
    )
pd.DataFrame(rows)

,video_id,gloss,set,signer,clip_frames,1-based length,0-based length,class train/val/test
0,18223,earring,train,20,10,9,10,8/3/2
1,59958,turkey,val,20,10,9,10,8/2/2
2,15144,deduct,val,20,9,8,9,5/2/1


## Share of each set

Instances `cutoff_9` removed, against the set sizes of the 1-based labels it was applied to.

In [3]:
share = []
for stem in split_stems:
    full = load_labels(f"{stem}_1_indexed")
    kept = load_labels(f"{stem}_cutoff_9_1_indexed")
    removed = full[~full.video_id.isin(kept.video_id)]
    for set_name in ("train", "val", "test"):
        n_set = int((full.set == set_name).sum())
        n_removed = int((removed.set == set_name).sum())
        share.append(
            {"split": stem, "set": set_name, "instances": n_set, "removed": n_removed, "share %": 100 * n_removed / n_set}
        )
pd.DataFrame(share).round(3)

,split,set,instances,removed,share %
0,asl1000,train,8978,1,0.011
1,asl1000,val,2320,1,0.043
2,asl1000,test,1876,0,0.000
3,asl2000,train,14296,1,0.007
4,asl2000,val,3920,2,0.051
5,asl2000,test,2879,0,0.000


## Scoring the clips with trained models

The removed clips are scored by every SATNAC 2026 checkpoint (S3D, MViTv2_S, MViTv2_S_e) and the
best MViTv2_B_32x3 run, for the splits whose `cutoff_9` removed them. The clips are loaded exactly as
the run's test set would load them (its own `test_augs`), once from the 1-based labels the models
were trained on and once from the 0-based labels. For a train clip (`18223`) this is an unseen
clip for these models, since they were trained without it.

Run once on the training server, then comment out.

In [4]:
# import tempfile
# from typing import TypedDict
#
# import torch
#
# from src.que.core import CompExpInfo, Que
# from src.results import stash_json
#
#
# class ClipScore(TypedDict):
#     split: str
#     model: str
#     exp_no: str
#     run_id: str
#     labels: str
#     video_id: str
#     gloss: str
#     target: int
#     pred: int
#     rank: int
#     true_prob: float
#     loss: float
#     best_val_acc: float
#     best_val_loss: float
#
#
# scored_runs = {
#     ("asl2000", "MViTv2_S_e", "000"), ("asl2000", "MViTv2_S", "002"), ("asl2000", "S3D", "050"),
#     ("asl2000", "S3D", "049"), ("asl2000", "MViTv2_B_32x3", "005"),
#     ("asl1000", "MViTv2_S_e", "000"), ("asl1000", "MViTv2_S", "002"), ("asl1000", "S3D", "049"),
#     ("asl1000", "S3D", "048"), ("asl1000", "MViTv2_B_32x3", "002"),
# }
#
#
# def score_run(run: CompExpInfo, label_dirs: dict[str, Path]) -> list[ClipScore]:
#     """Softmax scores of the removed clips for one run's best checkpoint, one entry per clip and
#     label convention (`label_dirs` maps convention -> a directory holding only those clips)."""
#     from src.models import get_model
#     from src.video_dataset import get_data_set, get_wlasl_info
#
#     stem = run.admin.split.split("_")[0]
#     model = get_model(run.admin.model, int(stem.removeprefix("asl")), drop_p=0.0)
#     model.load_state_dict(torch.load(Path(run.admin.save_path) / "checkpoints" / "best.pth"))
#     model.cuda().eval()
#     out: list[ClipScore] = []
#     for convention, label_dir in label_dirs.items():
#         info = get_wlasl_info(run.admin.split, "val")
#         info["labels"] = label_dir
#         dataset, _, _ = get_data_set(info, run.data)
#         for item, inst in zip((dataset[i] for i in range(len(dataset))), dataset.data):
#             with torch.no_grad():
#                 probs = torch.softmax(model(item["frames"].unsqueeze(0).cuda())[0].float(), dim=0)
#             target = int(item["label_num"])
#             out.append(
#                 ClipScore(
#                     split=stem, model=run.admin.model, exp_no=run.admin.exp_no,
#                     run_id=run.wandb.run_id, labels=convention,
#                     video_id=str(inst["video_id"]), gloss=str(inst["label_name"]),
#                     target=target, pred=int(probs.argmax()),
#                     rank=int((probs > probs[target]).sum()) + 1,
#                     true_prob=float(probs[target]), loss=float(-torch.log(probs[target])),
#                     best_val_acc=run.results.best_val_acc, best_val_loss=run.results.best_val_loss,
#                 )
#             )
#     return out
#
#
# def clip_label_dir(root: Path, stem: str, split: str, name: str) -> Path:
#     """Write a label dir holding only the removed clips of `split` (as a val set); return it."""
#     labels = load_labels(split)
#     clips = labels[labels.video_id.isin(clip_ids)].drop(columns="set")
#     out_dir = root / f"{stem}_{name}"
#     out_dir.mkdir(parents=True)
#     clips.to_json(out_dir / f"val_{LABEL_SUFFIX}", orient="records")
#     return out_dir
#
#
# scores: list[ClipScore] = []
# with tempfile.TemporaryDirectory() as tmp:
#     for stem in split_stems:
#         label_dirs = {
#             "1-based": clip_label_dir(Path(tmp), stem, f"{stem}_1_indexed", "one"),
#             "0-based": clip_label_dir(Path(tmp), stem, stem, "zero"),
#         }
#         for run in Que(auto_save=False).old_runs:
#             if (
#                 run.admin.split == f"{stem}_cutoff_9_1_indexed"
#                 and run.wandb.sweep_id is None
#                 and (stem, run.admin.model, run.admin.exp_no) in scored_runs
#             ):
#                 scores += score_run(run, label_dirs)
# stash_json(scores, scores_stash_p, indent=2)

In [5]:
scores = pd.DataFrame(load_json(scores_stash_p))
scores["run"] = scores.split + " " + scores.model + " " + scores.exp_no
scores["top1"] = scores.pred == scores.target
scores.groupby(["split", "labels", "video_id", "gloss"]).agg(
    runs=("top1", "size"),
    top1=("top1", "mean"),
    median_rank=("rank", "median"),
    mean_loss=("loss", "mean"),
).round(2)

runs  top1  median_rank  mean_loss
split   labels  video_id gloss                                      
asl1000 0-based 18223    earring     5   0.8          1.0       0.69
                59958    turkey      5   0.6          1.0       1.40
        1-based 18223    earring     5   0.6          1.0       1.24
                59958    turkey      5   0.8          1.0       1.56
asl2000 0-based 15144    deduct      5   0.6          1.0       1.08
                18223    earring     5   1.0          1.0       0.08
                59958    turkey      5   0.4          6.0       2.74
        1-based 15144    deduct      5   0.6          1.0       1.40
                18223    earring     5   1.0          1.0       0.13
                59958    turkey      5   0.4          8.0       2.88

In [6]:
scores[scores.labels == "1-based"].pivot_table(index="run", columns="video_id", values="rank")

video_id,15144,18223,59958
run,,,
asl1000 MViTv2_B_32x3 002,NaN,1.0,1.0
asl1000 MViTv2_S 002,NaN,2.0,1.0
asl1000 MViTv2_S_e 000,NaN,1.0,1.0
asl1000 S3D 048,NaN,2.0,1.0
asl1000 S3D 049,NaN,1.0,26.0
asl2000 MViTv2_B_32x3 005,1.0,1.0,8.0
asl2000 MViTv2_S 002,1.0,1.0,1.0
asl2000 MViTv2_S_e 000,1.0,1.0,1.0
asl2000 S3D 049,2.0,1.0,11.0


## What the clips would have done to validation

Adding a removed val clip back to a run's validation set changes its accuracy from `acc` to
`(acc * N + correct) / (N + k)`, for `N` val instances and `k` clips added, and its loss the same
way. `N` is the val size of the `cutoff_9` labels. The loss is approximate: `best_val_loss` is
the loss as logged during training, taken here as a per-instance mean. Only val clips matter:
`18223` is a train clip and none is in a test set.

In [7]:
val_n = {stem: int((load_labels(f"{stem}_cutoff_9_1_indexed").set == "val").sum()) for stem in split_stems}
val_clips = scores[(scores.labels == "1-based") & scores.video_id.isin(["59958", "15144"])]
impact = []
for (stem, run_name), g in val_clips.groupby(["split", "run"]):
    n, k = val_n[stem], len(g)
    acc, loss = g.best_val_acc.iloc[0], g.best_val_loss.iloc[0]
    impact.append(
        {
            "split": stem,
            "run": run_name.split(" ", 1)[1],
            "val clips": k,
            "val acc": acc,
            "acc with clips": 100 * (acc / 100 * n + g.top1.sum()) / (n + k),
            "val loss": loss,
            "loss with clips": (loss * n + g.loss.sum()) / (n + k),
        }
    )
impact = pd.DataFrame(impact).assign(
    acc_change=lambda d: d["acc with clips"] - d["val acc"], loss_change=lambda d: d["loss with clips"] - d["val loss"]
)
display(impact.round(4))
print(
    f"val acc change: {impact.acc_change.min():+.3f} to {impact.acc_change.max():+.3f} points; "
    f"val loss change: {impact.loss_change.min():+.4f} to {impact.loss_change.max():+.4f}; "
    f"one val instance is {100 / val_n['asl2000']:.3f} points of asl2000 val accuracy"
)

,split,run,val clips,val acc,acc with clips,val loss,loss with clips,acc_change,loss_change
0,asl1000,MViTv2_B_32x3 002,1,69.0384,69.0517,1.4591,1.4586,0.0133,-0.0005
1,asl1000,MViTv2_S 002,1,60.4571,60.4741,1.7126,1.7119,0.0170,-0.0007
2,asl1000,MViTv2_S_e 000,1,65.3299,65.3448,1.5315,1.5310,0.0149,-0.0005
3,asl1000,S3D 048,1,45.7094,45.7328,2.8848,2.8845,0.0234,-0.0003
4,asl1000,S3D 049,1,51.1427,51.1207,2.6088,2.6098,-0.0220,0.0011
5,asl2000,MViTv2_B_32x3 005,2,53.4967,53.4949,2.3369,2.3369,-0.0018,-0.0000
6,asl2000,MViTv2_S 002,2,46.8606,46.8878,2.6279,2.6270,0.0271,-0.0009
7,asl2000,MViTv2_S_e 000,2,50.7147,50.7398,2.4029,2.4019,0.0251,-0.0010
8,asl2000,S3D 049,2,35.2476,35.2296,3.8105,3.8104,-0.0180,-0.0002
9,asl2000,S3D 050,2,38.9995,38.9796,3.7856,3.7856,-0.0199,-0.0001


val acc change: -0.022 to +0.027 points; val loss change: -0.0010 to +0.0011; one val instance is 0.026 points of asl2000 val accuracy


## Can a training comparison see it?

The Que has no run to compare: the older asl1000 and asl2000 runs were all relabelled
`cutoff_9`, so none was trained with the clips in. The only configs run on both a plain and a
`cutoff_9` split are on asl100, where both splits hold the same 2038 instances (the cutoff removes
nothing there), so those are replicates, not a with/without comparison.

Replicates still give the run-to-run noise: the spread of val accuracy over runs of one config.
It is set against the effect above (at most 0.03 points).

Run once on the training server, then comment out.

In [8]:
# import statistics
# from collections import Counter, defaultdict
#
# from src.que.core import Que
# from src.results import stash_json
#
# runs_by_config = defaultdict(list)
# for run in Que(auto_save=False).old_runs:
#     if run.wandb.sweep_id is None:
#         key = (run.admin.model, run.admin.split.split("_")[0], run.admin.config_path.split("configfiles/")[-1])
#         runs_by_config[key].append(run)
#
# both = [k for k, v in runs_by_config.items() if {"cutoff" in r.admin.split for r in v} == {True, False}]
# replicates = []
# for (model, stem, config), rs in runs_by_config.items():
#     accs = [r.results.best_val_acc for r in rs if r.admin.split.endswith("cutoff_9_1_indexed")]
#     if len(accs) >= 3:
#         replicates.append(
#             {"model": model, "split": stem, "config": config, "runs": len(accs),
#              "val_acc_std": statistics.stdev(accs), "val_acc_range": max(accs) - min(accs)}
#         )
# stash_json(
#     {
#         "configs_on_plain_and_cutoff": [list(k) for k in both],
#         "plain_split_runs": dict(Counter(r.admin.split for rs in runs_by_config.values() for r in rs if "cutoff" not in r.admin.split)),
#         "replicates": replicates,
#     },
#     noise_stash_p,
#     indent=2,
# )

In [9]:
noise = load_json(noise_stash_p)
print("configs run on both a plain and a cutoff_9 split:", len(noise["configs_on_plain_and_cutoff"]), "(all on asl100)")
print("runs on plain splits:", noise["plain_split_runs"])
replicates = pd.DataFrame(noise["replicates"]).sort_values("runs", ascending=False)
display(replicates.head(8).round(2))
print(
    f"val acc std over replicates: median {replicates.val_acc_std.median():.2f} points "
    f"(range {replicates.val_acc_std.min():.2f}-{replicates.val_acc_std.max():.2f}), "
    f"{replicates.val_acc_std.median() / impact.acc_change.abs().max():.0f}x the largest effect of the clips"
)

configs run on both a plain and a cutoff_9 split: 4 (all on asl100)
runs on plain splits: {'asl100_1_indexed': 9, 'asl100_worst': 1, 'asl100_bottom': 1}


,model,split,config,runs,val_acc_std,val_acc_range
0,S3D,asl100,SeedComparison/S3D_czopef0v.toml,10,1.01,2.96
3,S3D,asl100,SeedComparison/S3D_13idpda6.toml,10,1.61,5.33
4,S3D,asl100,asl100/S3D/exp082.toml,5,1.51,3.55
2,MViTv2_S,asl100,asl100/MViTv2_S/exp010.toml,4,1.86,4.44
5,S3D,asl100,generic2/hframe_hwd_warmrestarts_hbs/tmult2.toml,4,1.70,3.55
1,S3D,asl100,asl100/S3D/exp101.toml,3,2.31,4.14


val acc std over replicates: median 1.65 points (range 1.01-2.31), 61x the largest effect of the clips


## Conclusions (as of 2026-10-15)

- **Removing the clips changes no test result.** None of the 5 is in a test set. The removed
  clips are 0.007-0.011% of a train set and 0.04-0.05% of a val set.
- **They would move validation by less than one instance.** Adding the val clips back changes the
  five runs' val accuracy by -0.022 to +0.027 points (one asl2000 val instance is 0.026) and val
  loss by about 0.001. The differences are of mixed sign, and far below the gaps between the models.
- **The models handle these clips like any others.** Of the 25 scores on the 1-based labels,
  17 are top-1 correct, a higher rate than the models' val accuracy (35-69%), so the clips are not
  outliers that hurt or help training. `59958` (turkey) is the hardest, with a median rank of 8
  on asl2000.
- **The effect of training with `18223` (earring) cannot be measured by training.** The Que has no
  with/without pair, since every asl1000 and asl2000 run used `cutoff_9`. Runs of one config differ
  by about 1-2 val accuracy points (median std 1.65), about 60 times the largest effect of the clips, so no feasible number of
  retrains would see it. The bound is one of 14296 train instances (and one of 8 for its class).
- **Label convention matters more than the cutoff.** The 1-based labels skip the first frame of
  every clip. The two conventions give similar scores on these 3 clips (mean loss differences have
  mixed sign), but that is 3 clips, not a comparison of the two conventions.

So a 0-based `cutoff_9` is not worth regenerating for its own sake. For the 0-based labels, which
the final results use, `cutoff_9` is not needed at all (see the decision in `src/TODO.md`).